In [ ]:
# ============================================================
# Statistical significance analysis
# Wilcoxon + Effect Size + Holm Correction
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

DRIVE_DIR = "/content/drive/MyDrive/iTransformer"
OUTPUT_DIR = f"{DRIVE_DIR}/model_outputs"

TARGET_COLS = [
    "TARGET_threshold_hours",
    "TARGET_threshold_inactive_min",
    "TARGET_threshold_fluctuation",
    "TARGET_each_hour_fluctuation",
]

# ------------------------------------------------------------
# Load ground truth
# ------------------------------------------------------------

y_true = pd.read_csv(f"{DRIVE_DIR}/y_test.csv")

# ------------------------------------------------------------
# Load predictions
# ------------------------------------------------------------

pred1 = pd.read_csv(f"{OUTPUT_DIR}/model1_predictions.csv")
pred2 = pd.read_csv(f"{OUTPUT_DIR}/model2_predictions.csv")
pred3 = pd.read_csv(f"{OUTPUT_DIR}/model3_predictions.csv")
pred4 = pd.read_csv(f"{OUTPUT_DIR}/model4_predictions.csv")

models = {
    "XGBoost": pred1,
    "CNN-BiLSTM": pred2,
    "PCTN": pred3,
    "iTransformer": pred4,
}

# ------------------------------------------------------------
# Rank-biserial correlation
# ------------------------------------------------------------

def rank_biserial_from_wilcoxon(x, y):
    """
    Rank-biserial correlation for paired samples.

    Returns value between -1 and 1.
    """

    d = x - y
    d = d[d != 0]

    if len(d) == 0:
        return np.nan

    abs_d = np.abs(d)

    order = abs_d.argsort()
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(abs_d)+1)

    pos = ranks[d > 0].sum()
    neg = ranks[d < 0].sum()

    return (pos - neg) / (pos + neg)

# ------------------------------------------------------------
# Pairwise tests
# ------------------------------------------------------------

pairs = [
    ("PCTN", "XGBoost"),
    ("PCTN", "CNN-BiLSTM"),
    ("PCTN", "iTransformer"),
]

results = []

for target in TARGET_COLS:

    y = y_true[target].values

    for A, B in pairs:

        errA = np.abs(y - models[A][target + "_pred"].values)
        errB = np.abs(y - models[B][target + "_pred"].values)

        stat, p = wilcoxon(
            errA,
            errB,
            alternative="two-sided",
            zero_method="wilcox"
        )

        effect = rank_biserial_from_wilcoxon(errA, errB)

        results.append({
            "Target": target.replace("TARGET_", ""),
            "Comparison": f"{A} vs {B}",
            "Statistic": stat,
            "p_raw": p,
            "EffectSize_rbc": effect
        })

# ------------------------------------------------------------
# Holm correction
# ------------------------------------------------------------

pvals = [r["p_raw"] for r in results]

holm = multipletests(
    pvals,
    alpha=0.05,
    method="holm"
)

bonf = multipletests(
    pvals,
    alpha=0.05,
    method="bonferroni"
)

for i in range(len(results)):
    results[i]["p_holm"] = holm[1][i]
    results[i]["holm_sig"] = holm[0][i]

    results[i]["p_bonf"] = bonf[1][i]
    results[i]["bonf_sig"] = bonf[0][i]

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------

df = pd.DataFrame(results)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print(df)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

outfile = f"{OUTPUT_DIR}/wilcoxon_effectsize_results.csv"

df.to_csv(outfile, index=False)

print("\nSaved:")
print(outfile)

                    Target            Comparison     Statistic  p_raw  EffectSize_rbc  p_holm  holm_sig  p_bonf  bonf_sig
0          threshold_hours       PCTN vs XGBoost  8.576269e+08    0.0       -0.759465     0.0      True     0.0      True
1          threshold_hours    PCTN vs CNN-BiLSTM  1.738301e+09    0.0        0.315081     0.0      True     0.0      True
2          threshold_hours  PCTN vs iTransformer  2.214288e+08    0.0       -0.951488     0.0      True     0.0      True
3   threshold_inactive_min       PCTN vs XGBoost  8.275260e+05    0.0       -0.999819     0.0      True     0.0      True
4   threshold_inactive_min    PCTN vs CNN-BiLSTM  1.660848e+09    0.0        0.231269     0.0      True     0.0      True
5   threshold_inactive_min  PCTN vs iTransformer  2.608839e+09    0.0       -0.428436     0.0      True     0.0      True
6    threshold_fluctuation       PCTN vs XGBoost  2.196262e+09    0.0        0.518776     0.0      True     0.0      True
7    threshold_fluctuati

In [ ]:
# ------------------------------------------------------------
# Pairwise tests
# Wilcoxon + standardized effect size r = |Z| / sqrt(N)
# ------------------------------------------------------------

from scipy.stats import norm

pairs = [
    ("PCTN", "XGBoost"),
    ("PCTN", "CNN-BiLSTM"),
    ("PCTN", "iTransformer"),
]

results = []

for target in TARGET_COLS:

    y = y_true[target].values

    for A, B in pairs:

        errA = np.abs(y - models[A][target + "_pred"].values)
        errB = np.abs(y - models[B][target + "_pred"].values)

        stat, p = wilcoxon(
            errA,
            errB,
            alternative="two-sided",
            zero_method="wilcox"
        )

        # Number of non-zero paired differences
        diff = errA - errB
        n = np.sum(diff != 0)

        # Expected value and standard deviation of W
        mean_W = n * (n + 1) / 4
        sd_W = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)

        # Standardized Z
        z = (stat - mean_W) / sd_W

        # Effect size
        r = abs(z) / np.sqrt(n)

        results.append({
            "Target": target.replace("TARGET_", ""),
            "Comparison": f"{A} vs {B}",
            "Statistic": stat,
            "Z": z,
            "EffectSize_r": r,
            "p_raw": p,
        })

# ------------------------------------------------------------
# Holm and Bonferroni correction
# ------------------------------------------------------------

pvals = [r["p_raw"] for r in results]

holm = multipletests(
    pvals,
    alpha=0.05,
    method="holm"
)

bonf = multipletests(
    pvals,
    alpha=0.05,
    method="bonferroni"
)

for i in range(len(results)):
    results[i]["p_holm"] = holm[1][i]
    results[i]["holm_sig"] = holm[0][i]
    results[i]["p_bonf"] = bonf[1][i]
    results[i]["bonf_sig"] = bonf[0][i]

df = pd.DataFrame(results)

print(df)

outfile = f"{OUTPUT_DIR}/wilcoxon_effectsize_results.csv"
df.to_csv(outfile, index=False)

print(f"\nSaved to: {outfile}")

                    Target            Comparison     Statistic           Z  EffectSize_r  p_raw  p_holm  holm_sig  p_bonf  bonf_sig
0          threshold_hours       PCTN vs XGBoost  8.576269e+08 -227.291469      0.657717    0.0     0.0      True     0.0      True
1          threshold_hours    PCTN vs CNN-BiLSTM  1.738301e+09  -86.614381      0.272869    0.0     0.0      True     0.0      True
2          threshold_hours  PCTN vs iTransformer  2.214288e+08 -302.896636      0.824014    0.0     0.0      True     0.0      True
3   threshold_inactive_min       PCTN vs XGBoost  8.275260e+05 -318.282315      0.865870    0.0     0.0      True     0.0      True
4   threshold_inactive_min    PCTN vs CNN-BiLSTM  1.660848e+09  -61.066408      0.200286    0.0     0.0      True     0.0      True
5   threshold_inactive_min  PCTN vs iTransformer  2.608839e+09 -136.388335      0.371037    0.0     0.0      True     0.0      True
6    threshold_fluctuation       PCTN vs XGBoost  2.196262e+09 -165.143100  